 # Scenario: "The Environmental Policy Compliance Assistant"
Background
You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
The company must ensure compliance to avoid fines and reputational damage.

Challenge
The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
Your task is to:
- Process the regulation document so the assistant can store and understand it.
- Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
- Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

Roles
- Learner (You): Environmental compliance officer using the assistant.
- Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
- Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

🔄 Flow of the Scenario
- Upload Environmental Regulation PDF
Example: “National Carbon Emissions Act 2026”.
- System Processes Document
- Splits into chunks.
- Embeds into vector database.
- Stores for retrieval.
- Ask Questions
- “What is the maximum carbon emission allowed per factory per year?”
- “What penalties apply if hazardous waste is not disposed of properly?”
- “What renewable energy targets must we meet by 2030?”
- Assistant Responds
- Retrieves relevant chunks.
- Generates compliance-focused answers.
- Provides short, clear guidance.
- Outcome
- Learners practice extracting environmental obligations.
- Managers receive summarized compliance insights.
- Executives gain confidence in sustainability strategy alignment.

🎯 Training Objective
This scenario helps learners:
- Understand how RAG systems can support environmental compliance.
- Practice formulating precise queries to extract obligations.
- Experience how AI can simplify complex environmental regulations into actionable steps.

👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?

In [1]:
# ============================================================
# ENVIRONMENTAL POLICY COMPLIANCE ASSISTANT (RAG SYSTEM)
# ============================================================

# ------------------------------------------------------------
# STEP 1 — Install Dependencies
# ------------------------------------------------------------

!pip install -q gradio chromadb sentence-transformers pypdf transformers accelerate torch


# ------------------------------------------------------------
# STEP 2 — Import Libraries
# ------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ------------------------------------------------------------
# STEP 3 — Load Embedding Model
# ------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# ------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("environmental_docs")
except:
    pass

collection = client.create_collection("environmental_docs")


# ------------------------------------------------------------
# STEP 5 — Load Language Model
# ------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=200
)

print("LLM loaded successfully")


# ------------------------------------------------------------
# STEP 6 — Text Chunking
# ------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


# ------------------------------------------------------------
# STEP 7 — Process Environmental Regulation PDF
# ------------------------------------------------------------

def process_pdf(file):

    if file is None:
        return "Please upload an environmental regulation PDF."

    print("Processing Environmental Regulation...")

    reader = PdfReader(file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    if text.strip() == "":
        return "No readable text found in the PDF."

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Environmental regulation processed. {len(chunks)} sections stored."


# ------------------------------------------------------------
# STEP 8 — Retrieve Relevant Sections
# ------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    return docs


# ------------------------------------------------------------
# STEP 9 — Generate Compliance Answer
# ------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are an Environmental Compliance Assistant.

Use ONLY the regulation context below.

Context:
{context}

Question:
{query}

Provide a short compliance-focused answer.
"""

    response = llm(prompt)

    return response[0]["generated_text"]


# ------------------------------------------------------------
# STEP 10 — Chat Function
# ------------------------------------------------------------

def chat(question):

    if question is None or question.strip() == "":
        return "Please enter a compliance question."

    return answer_question(question)


# ------------------------------------------------------------
# STEP 11 — Gradio Interface
# ------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🌍 Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload an environmental regulation PDF and ask questions about:

* Carbon emission limits
* Waste disposal rules
* Renewable energy targets
* Environmental penalties
""")

    pdf_input = gr.File(file_types=[".pdf"])

    upload_button = gr.Button("Process Regulation Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask an Environmental Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Assistant Response",
        lines=10
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# ------------------------------------------------------------
# STEP 12 — Launch Application
# ------------------------------------------------------------

demo.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded successfully
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://caf280bedb9e28d2de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
